# doc-extraction — OmniDocBench benchmark (Kaggle, T4 GPU)

Thin orchestrator notebook — no extraction or evaluation logic lives here.
Everything runs through the repo's own scripts
(`experiments/005_omnidocbench/run.py`,
`src/doc_extraction/evaluation/omnidocbench.py`). See
[`experiments/005_omnidocbench/README.md`](README.md) for the full design
and the pinned OmniDocBench commit.

**No private data.** Only the public OmniDocBench dataset is used here.
This repo's own `data/` (local/private sample documents) is gitignored and
is not part of this clone — never attach it as a Kaggle input.

## Two separate Python environments, on purpose

```
Kaggle kernel (Python 3.12)
    doc-extraction pipeline  ->  predictions (GPU-capable)
        |
        v  subprocess
Isolated venv (Python 3.11, built with uv)
    official OmniDocBench evaluator  ->  metrics (CPU-only)
```

The pinned evaluator requires Python `>=3.10,<3.12`; Kaggle's kernel is
3.12+. The evaluator always runs as an external subprocess in its own venv —
never imported into the main kernel, its source never modified, nothing
monkey-patched onto `sys.path`.

## GPU status (changed — read this)

Earlier runs of this notebook wasted the T4. Two real bugs, both now fixed
in the repo (not worked around here):

1. `DoclingBackend` never set `pipeline_options.accelerator_options`, and
   passed the deprecated `EasyOcrOptions(use_gpu=...)` derived from
   `config.device`. With `configs/cpu.yaml` that pinned **EasyOCR to CPU**
   while docling's own `device="auto"` default silently put layout on CUDA —
   an inconsistent split, and OCR is the expensive stage on scanned pages.
   That is what the per-page `UserWarning: Deprecated field...` was telling
   us, and why layout-stage wall time sat at ~28 s/page on a T4.
2. `TableTransformerBackend` moved its *models* to `self.device` but never
   its *inputs*, so `device: cuda` would have crashed with "Expected all
   tensors to be on the same device".

Section 4 below now selects `configs/gpu.yaml` when CUDA is present, and one
setting drives every model stage. Section 8 measures seconds/page so the
speedup is **observed, not assumed** — do not claim a GPU win from this
notebook without reading that number.

In [25]:
print("==================================================")
print("1. RUNTIME")
print("==================================================")

1. RUNTIME


In [26]:
import sys
print("Kernel Python:", sys.version)
assert sys.version_info >= (3, 10), "doc_extraction needs Python 3.10+"

Kernel Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [27]:
import torch

CUDA_AVAILABLE = torch.cuda.is_available()
print(f"torch           : {torch.__version__}")
print(f"CUDA available  : {CUDA_AVAILABLE}")
if CUDA_AVAILABLE:
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version    : {torch.version.cuda}")
    print(f"VRAM GB         : {round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)}")
else:
    print("No GPU visible — Settings > Accelerator should be GPU T4 x2 (or T4 x1).")
    print("The pipeline will run CPU-only: expect roughly 28 s/page.")

torch           : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
CUDA version    : 12.8
VRAM GB         : 14.56


In [28]:
print("==================================================")
print("2. REPOSITORY")
print("==================================================")

2. REPOSITORY


Idempotent: reruns reuse an existing clone instead of failing on `git
clone` into a non-empty directory, and pull so a stale clone from an
earlier session doesn't shadow the GPU fixes described above.

In [29]:
import os

REPO_DIR = "/kaggle/working/doc-extraction"

if not os.path.exists(REPO_DIR):
    os.chdir("/kaggle/working")
    !git clone https://github.com/anhkhoa1804/doc-extraction.git
else:
    print("Repository already exists:", REPO_DIR, "- pulling latest")
    !git -C {REPO_DIR} pull --ff-only

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!git rev-parse HEAD

Repository already exists: /kaggle/working/doc-extraction - pulling latest
Already up to date.
cwd: /kaggle/working/doc-extraction
a5c0420087e90b8a650c8b8ab92d1f9a1fbe1adc


In [30]:
print("==================================================")
print("3. DOC-EXTRACTION ENVIRONMENT")
print("==================================================")

3. DOC-EXTRACTION ENVIRONMENT


The main project, installed into the Kaggle kernel's own Python 3.12 — no
isolated venv needed here, its dependencies have no `<3.12` ceiling.

In [31]:
%pip install -e ".[docling,tables]"

Obtaining file:///kaggle/working/doc-extraction
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for doc-extraction (pyproject.toml) ... done
  Created wheel for doc-extraction: filename=doc_extraction-0.1.0-py3-none-any.whl size=5088 sha256=adf02a85b93e39812f700f26aecda2dbe668630368958f71a5d41208232c993c
  Stored in directory: /tmp/pip-ephem-wheel-cache-m5u920pc/wheels/1f/92/9e/8ec7e165555e59d1e824adef614c990b5715210ec843fbd769
Successfully built doc-extraction
  Attempting uninstall: doc-extraction
    Found existing installation: doc-extraction 0.1.0
    Uninstalling doc-extraction-0.1.0:
      Successfully uninstalled doc-extraction-0.1.0
Note: you may need to restart the kernel to use updated packages.


In [32]:
import doc_extraction
print("doc_extraction imported OK:", doc_extraction.__file__)

doc_extraction imported OK: /kaggle/working/doc-extraction/src/doc_extraction/__init__.py


**Prefetch Docling models — required on Kaggle, not optional.** Once
`DOCLING_ARTIFACTS_PATH` is set to anything, Docling stops auto-downloading
and requires that directory to already hold the models, raising
`RuntimeError: ... is not valid` otherwise. Prefetch once, up front.

In [33]:
import os

CACHE_ROOT = "/kaggle/working/.cache"
os.makedirs(f"{CACHE_ROOT}/huggingface", exist_ok=True)
os.makedirs(f"{CACHE_ROOT}/docling", exist_ok=True)
os.environ["HF_HOME"] = f"{CACHE_ROOT}/huggingface"
os.environ["DOCLING_ARTIFACTS_PATH"] = f"{CACHE_ROOT}/docling"
os.environ["XDG_CACHE_HOME"] = CACHE_ROOT

# Default model set (layout, TableFormer, ...) plus EasyOCR for the en/vi
# languages the configs declare. Note: OmniDocBench also contains Chinese
# pages, which these EasyOCR language models do not cover — a pre-existing
# config choice inherited from this repo's own corpus, not set here.
!python -m docling.cli.tools models download -o {CACHE_ROOT}/docling
!python -m docling.cli.tools models download easyocr \
    --easyocr-lang en --easyocr-lang vi -o {CACHE_ROOT}/docling

HTTP Request: GET                                                ]8;id=135745;file:///usr/local/lib/python3.12/dist-packages/httpx/_client.py\_client.py]8;;\:]8;id=506151;file:///usr/local/lib/python3.12/dist-packages/httpx/_client.py#1025\1025]8;;\
https://huggingface.co/api/models/docling-project/docling-layout                
-heron/revision/main "HTTP/1.1 200 OK"                                          
Fetching 6 files: 100%|█████████████████████████| 6/6 [00:00<00:00, 1164.92it/s]
Download complete: : 0.00B [00:00, ?B/s]              HTTP Request: GET                                                ]8;id=647811;file:///usr/local/lib/python3.12/dist-packages/httpx/_client.py\_client.py]8;;\:]8;id=812186;file:///usr/local/lib/python3.12/dist-packages/httpx/_client.py#1025\1025]8;;\
https://huggingface.co/api/models/docling-project/docling-layout                
-heron-onnx/revision/main "HTTP/1.1 200 OK"                                     


Fetching 5 files: 100%|

In [34]:
print("==================================================")
print("4. PIPELINE DEVICE CONFIG")
print("==================================================")

4. PIPELINE DEVICE CONFIG


Picks the pipeline config from what the runtime actually has, and prints
the resolved device so the choice is visible rather than implied.

`configs/gpu.yaml` sets `device: cuda`, which now propagates to **every**
model stage — docling layout, TableFormer, and EasyOCR — via
`AcceleratorOptions`, plus the Table Transformer backend. Before the fix
this file was documented as unvalidated and would in fact have crashed the
table stage; see the intro. If CUDA is absent this falls back to
`configs/cpu.yaml`, the profile this project has always validated.

In [35]:
PIPELINE_CONFIG = "configs/gpu.yaml" if CUDA_AVAILABLE else "configs/cpu.yaml"
print(f"PIPELINE_CONFIG = {PIPELINE_CONFIG}")

import sys
sys.path.insert(0, f"{REPO_DIR}/src")
from doc_extraction.config import load_config

_cfg = load_config(PIPELINE_CONFIG)
print(f"device          = {_cfg.device}")
print(f"ocr_languages   = {_cfg.ocr_languages}")
print(f"layout_backend  = {_cfg.layout_backend}")
print(f"table_backend   = {_cfg.table_backend}")

if _cfg.device == "cuda" and not CUDA_AVAILABLE:
    raise RuntimeError("config requests cuda but torch.cuda.is_available() is False")

PIPELINE_CONFIG = configs/gpu.yaml
device          = cuda
ocr_languages   = ['en', 'vi']
layout_backend  = docling
table_backend   = table_transformer


In [36]:
print("==================================================")
print("5. OMNIDOCBENCH EVALUATOR ENVIRONMENT")
print("==================================================")

5. OMNIDOCBENCH EVALUATOR ENVIRONMENT


**Why this kept failing with `ModuleNotFoundError: No module named 'yaml'`.**
The evaluator's own `pyproject.toml` declares `PyYAML==6.0.2`, so the
dependency was never actually missing from the spec. The install simply
never landed in the venv: Kaggle's image sets **`UV_SYSTEM_PYTHON`**, which
makes `uv pip install --python <venv>/bin/python` ignore the virtualenv
entirely and resolve to uv's own managed base interpreter — which then
refuses the install as *"externally managed"*. The tell-tale evidence was
in the log all along:

```
warning: The `--system` flag has no effect, `uv venv` always ignores ...
error: The interpreter at /root/.local/share/uv/python/cpython-3.11...
       is externally managed
hint: Virtual environments were not considered due to the `--system` flag
```

Neither cell passed `--system`; the environment did. So the venv stayed
empty and `run.py` later hit a `yaml`-less interpreter.

The fix removes the ambiguity entirely rather than reinstalling PyYAML:
scrub `UV_SYSTEM_PYTHON` from the env handed to `uv`, create the venv with
`--seed` so it owns a `pip`, then install via `<venv>/bin/python -m pip`,
which targets that exact interpreter *by construction* — there is no
`--python` path for uv to re-resolve.

In [37]:
import os
import subprocess

OMNIDOC_REPO = f"{REPO_DIR}/.external/OmniDocBench"
PINNED_OMNIDOC_COMMIT = "193627ae9e97d89188468ed1ee3b7a856ff76044"

if not os.path.exists(OMNIDOC_REPO):
    os.makedirs(f"{REPO_DIR}/.external", exist_ok=True)
    !git clone https://github.com/opendatalab/OmniDocBench.git {OMNIDOC_REPO}
!git -C {OMNIDOC_REPO} fetch --all --tags -q
!git -C {OMNIDOC_REPO} checkout {PINNED_OMNIDOC_COMMIT}

_actual = subprocess.run(
    ["git", "-C", OMNIDOC_REPO, "rev-parse", "HEAD"],
    capture_output=True, text=True, check=True,
).stdout.strip()
assert _actual == PINNED_OMNIDOC_COMMIT, (
    f"OmniDocBench is at {_actual}, expected pinned {PINNED_OMNIDOC_COMMIT}"
)
print(f"OmniDocBench pinned at {PINNED_OMNIDOC_COMMIT}")

HEAD is now at 193627a Merge pull request #250 from Yunnglin/evalscope-omnidocbench-docs-20260727-103734
OmniDocBench pinned at 193627ae9e97d89188468ed1ee3b7a856ff76044


In [38]:
# Absolute paths throughout — a relative '.venv-omnidoc/bin/python' resolves
# somewhere different the moment a cell's cwd differs.
OMNIDOC_VENV = f"{REPO_DIR}/.venv-omnidoc"
OMNIDOC_PYTHON = f"{OMNIDOC_VENV}/bin/python"

print("OMNIDOC_VENV  :", OMNIDOC_VENV)
print("OMNIDOC_PYTHON:", OMNIDOC_PYTHON)

OMNIDOC_VENV  : /kaggle/working/doc-extraction/.venv-omnidoc
OMNIDOC_PYTHON: /kaggle/working/doc-extraction/.venv-omnidoc/bin/python


In [39]:
import json
import os
import shutil
import subprocess

# uv must not inherit Kaggle's UV_SYSTEM_PYTHON (see the markdown above),
# nor a VIRTUAL_ENV pointing anywhere else.
UV_ENV = {k: v for k, v in os.environ.items()
          if k not in ("UV_SYSTEM_PYTHON", "VIRTUAL_ENV")}


def check_evaluator_health():
    """Probe runs *inside* OMNIDOC_PYTHON, so it verifies what run.py will
    actually invoke - not what uv or pip merely claim about some env."""
    if not os.path.exists(OMNIDOC_PYTHON):
        return False, f"no interpreter at {OMNIDOC_PYTHON}"

    probe = (
        "import sys, json; "
        "info = {'executable': sys.executable, 'prefix': sys.prefix, "
        "        'base_prefix': sys.base_prefix, 'version': list(sys.version_info[:2])}; "
        "import yaml; info['yaml_version'] = yaml.__version__; "
        "from src.core.pipeline import run_config_file; info['entrypoint'] = 'OK'; "
        "print(json.dumps(info))"
    )
    r = subprocess.run([OMNIDOC_PYTHON, "-c", probe], cwd=OMNIDOC_REPO,
                       capture_output=True, text=True)
    if r.returncode != 0:
        return False, "\n".join((r.stderr or r.stdout).strip().splitlines()[-12:])

    try:
        info = json.loads(r.stdout.strip().splitlines()[-1])
    except (ValueError, IndexError) as exc:
        return False, f"unparseable probe output: {exc}; raw={r.stdout!r}"

    major, minor = info["version"]
    if not (major == 3 and 10 <= minor < 12):
        return False, f"interpreter is Python {major}.{minor}, need 3.10/3.11 (<3.12)"

    # Identity is checked via sys.prefix, NOT realpath(sys.executable).
    # uv symlinks the venv's bin/python at its managed base install, so
    # resolving symlinks reports a path outside the venv even when the venv
    # is perfectly correct. sys.prefix is the venv root by definition.
    def _norm(p):
        return os.path.normpath(os.path.abspath(p))

    if _norm(info["prefix"]) != _norm(OMNIDOC_VENV):
        return False, (f"interpreter's sys.prefix={info['prefix']!r} is not "
                       f"the expected venv {OMNIDOC_VENV!r}")
    if info["prefix"] == info["base_prefix"]:
        return False, (f"{OMNIDOC_PYTHON} is not running inside a virtualenv "
                       f"(sys.prefix == sys.base_prefix)")
    return True, info


def build_evaluator_venv():
    print(f"Building evaluator venv at {OMNIDOC_VENV} ...")
    if os.path.exists(OMNIDOC_VENV):
        shutil.rmtree(OMNIDOC_VENV)

    subprocess.run(["uv", "python", "install", "3.11"], check=True, env=UV_ENV)
    # --seed puts pip/setuptools/wheel *inside* the venv so the install below
    # can be driven by the venv's own interpreter.
    subprocess.run(["uv", "venv", "--python", "3.11", "--seed", OMNIDOC_VENV],
                   check=True, env=UV_ENV)

    # Primary: the venv's own interpreter installs into itself. Installs the
    # evaluator plus every dependency its pyproject.toml declares (PyYAML
    # included) - upstream's declaration is the source of truth, never a
    # hand-picked subset.
    primary = subprocess.run(
        [OMNIDOC_PYTHON, "-m", "pip", "install", "--no-input", "-q", "-e", OMNIDOC_REPO],
        capture_output=True, text=True,
    )
    if primary.returncode == 0:
        print("  installed via the venv's own pip")
        return

    print("  venv pip failed, falling back to uv with a scrubbed env:")
    print("  " + "\n  ".join((primary.stderr or "").strip().splitlines()[-8:]))
    subprocess.run(
        ["uv", "pip", "install", "--python", OMNIDOC_PYTHON, "-e", OMNIDOC_REPO],
        check=True, env=UV_ENV,
    )

In [40]:
!pip install -q -U uv

ok, info = check_evaluator_health()
if not ok:
    print(f"Evaluator env not healthy:\n{info}\n")
    build_evaluator_venv()
    ok, info = check_evaluator_health()

assert ok, f"Evaluator venv still unhealthy after a clean rebuild:\n{info}"

EVALUATOR_READY = True
print("Evaluator environment OK")
print(f"  interpreter : {info['executable']}")
print(f"  version     : Python {info['version'][0]}.{info['version'][1]}")
print(f"  yaml        : {info['yaml_version']}")
print(f"  entrypoint  : {info['entrypoint']}")

Evaluator environment OK
  interpreter : /kaggle/working/doc-extraction/.venv-omnidoc/bin/python
  version     : Python 3.11
  yaml        : 6.0.2
  entrypoint  : OK


In [41]:
print("==================================================")
print("6. DATASET")
print("==================================================")

6. DATASET


**Option A (the real benchmark): attach a Kaggle Dataset** holding
`OmniDocBench.json` + `images/` (from
https://huggingface.co/datasets/opendatalab/OmniDocBench) via "Add Input",
then set `KAGGLE_DATASET_NAME` below to match.

**Option B (smoke/demo only): the bundled 18-page official demo set.** Never
silently treated as the full benchmark — `USING_FULL_DATASET` is `False`
while it's in use, and the full-benchmark cells refuse to run.

Either way, only public OmniDocBench data — never this repo's own `data/`.

In [42]:
import os

# Edit this to the Kaggle Dataset you attached via "Add Input".
KAGGLE_DATASET_NAME = "<dataset-name>"

_attached = f"/kaggle/input/{KAGGLE_DATASET_NAME}"
_demo = f"{OMNIDOC_REPO}/demo_data/omnidocbench_demo"

USING_FULL_DATASET = os.path.exists(_attached)
DATASET_PATH = _attached if USING_FULL_DATASET else _demo
OUTPUT_ROOT = "/kaggle/working/results"

if USING_FULL_DATASET:
    print("Attached Kaggle Dataset found — using the FULL dataset.")
else:
    print(f"No Kaggle Dataset attached at {_attached}.")
    print("Using the 18-page OFFICIAL DEMO SET — smoke/demo testing only.")
    print("The full-benchmark cells will refuse to run until a real dataset")
    print("is attached and KAGGLE_DATASET_NAME is set to match it.")

print()
print(f"USING_FULL_DATASET = {USING_FULL_DATASET}")
print(f"DATASET_PATH       = {DATASET_PATH}")

No Kaggle Dataset attached at /kaggle/input/<dataset-name>.
Using the 18-page OFFICIAL DEMO SET — smoke/demo testing only.
The full-benchmark cells will refuse to run until a real dataset
is attached and KAGGLE_DATASET_NAME is set to match it.

USING_FULL_DATASET = False
DATASET_PATH       = /kaggle/working/doc-extraction/.external/OmniDocBench/demo_data/omnidocbench_demo


In [43]:
assert os.path.exists(DATASET_PATH), (
    f"DATASET_PATH does not exist: {DATASET_PATH} — if this is the demo-set "
    f"fallback, run the EVALUATOR ENVIRONMENT section first (it clones "
    f".external/OmniDocBench, which the demo set lives inside)."
)
print(f"DATASET_PATH = {DATASET_PATH}")
print(sorted(os.listdir(DATASET_PATH))[:20])

DATASET_PATH = /kaggle/working/doc-extraction/.external/OmniDocBench/demo_data/omnidocbench_demo
['OmniDocBench_demo.json', 'images', 'mds']


In [44]:
print("==================================================")
print("7. EXPERIMENT CLI")
print("==================================================")

7. EXPERIMENT CLI


The current CLI surface, printed rather than assumed, so the commands
below match the flags that really exist.

In [45]:
!python experiments/005_omnidocbench/run.py --help

usage: run.py [-h] --dataset DATASET --backend {baseline,docling} --output
              OUTPUT [--config CONFIG] [--subset SUBSET] [--seed SEED]
              [--omnidoc-repo OMNIDOC_REPO] [--omnidoc-python OMNIDOC_PYTHON]
              [--match-method {no_split,simple_match,quick_match}]
              [--match-workers MATCH_WORKERS] [--include-bleu-meteor]
              [--include-cdm] [--defaults-config DEFAULTS_CONFIG]
              [--skip-prepare] [--skip-evaluate]

OmniDocBench experiment orchestrator: prepare (generate predictions) then
evaluate (invoke the official evaluator), then write a human-readable
report.md. Thin wrapper around prepare.py + evaluate.py — see those for the
independently-rerunnable steps this calls.

    python experiments/005_omnidocbench/run.py \
        --dataset /path/to/OmniDocBench \
        --backend baseline \
        --output experiments/005_omnidocbench/results/baseline

    python experiments/005_omnidocbench/run.py \
        --dataset /path/to

In [46]:
print("==================================================")
print("8. SMOKE TEST (predictions only)")
print("==================================================")

8. SMOKE TEST (predictions only)


Predictions only (`--skip-evaluate`) for a small, deterministic 3-page
subset. Scoring is the next section, kept separate so an evaluator failure
never forces regenerating predictions to retry.

The cell after this reads the measured seconds/page back out of
`runtime.json` and compares it to the CPU-only reference — that number, not
this notebook's prose, is the evidence for whether the GPU is doing work.

In [47]:
!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend baseline \
    --output {OUTPUT_ROOT}/baseline_smoke \
    --config {PIPELINE_CONFIG} \
    --subset 3 \
    --omnidoc-python {OMNIDOC_PYTHON} \
    --skip-evaluate

dataset: /kaggle/working/doc-extraction/.external/OmniDocBench/demo_data/omnidocbench_demo/OmniDocBench_demo.json
samples: 3 of 18 available (backend=baseline)
09:10:54 [INFO] [yanbaopptmerge_SE05.pdf_7-660a2591] render backend=pymupdf page=0 status=success runtime=0.26834246599901235
Loading weights: 100%|██████████████████████| 770/770 [00:00<00:00, 2851.98it/s]
09:11:13 [INFO] [yanbaopptmerge_SE05.pdf_7-660a2591] layout backend=docling page=0 status=success runtime=19.166848021999613
09:11:13 [INFO] [yanbaopptmerge_SE05.pdf_7-660a2591] ocr backend=docling page=0 status=success runtime=0.0007183340003393823
Loading weights: 100%|██████████████████████| 367/367 [00:00<00:00, 2745.69it/s]
[transformers] TableTransformerForObjectDetection LOAD REPORT from: microsoft/table-transformer-detection
Key                                                                         | Status     |  | 
----------------------------------------------------------------------------+------------+--+-
model.

In [48]:
import json
from pathlib import Path

# Measured on Kaggle's T4 with the pre-fix code, where EasyOCR was pinned to
# CPU: 69.1 s for the first page (model load) then ~28 s/page steady state.
CPU_REFERENCE_SECONDS_PER_PAGE = 28.0

rt = json.loads(Path(f"{OUTPUT_ROOT}/baseline_smoke/runtime.json").read_text())
mean = rt.get("mean_seconds_per_page")

print(f"config           : {PIPELINE_CONFIG} (device={_cfg.device})")
print(f"pages ok/failed  : {rt.get('succeeded')}/{rt.get('failed')}")
print(f"mean s/page      : {mean}")
print(f"CPU reference    : {CPU_REFERENCE_SECONDS_PER_PAGE} s/page (pre-fix, EasyOCR on CPU)")

if mean:
    print(f"ratio vs CPU ref : {CPU_REFERENCE_SECONDS_PER_PAGE / mean:.2f}x")
    print()
    print("Note: the first page includes one-time model loading, so a 3-page")
    print("mean understates steady-state throughput. Read the per-page lines")
    print("in the log above for the real per-page cost.")

config           : configs/gpu.yaml (device=cuda)
pages ok/failed  : 3/0
mean s/page      : 14.0843
CPU reference    : 28.0 s/page (pre-fix, EasyOCR on CPU)
ratio vs CPU ref : 1.99x

Note: the first page includes one-time model loading, so a 3-page
mean understates steady-state throughput. Read the per-page lines
in the log above for the real per-page cost.


In [49]:
print("==================================================")
print("9. EVALUATION")
print("==================================================")

9. EVALUATION


Scores the smoke-test predictions with `--skip-prepare`, reusing
`/kaggle/working/results/baseline_smoke/predictions/` as-is rather than
regenerating them.

**If this fails**, re-run the EVALUATOR ENVIRONMENT section (it rebuilds the
venv from scratch when the health check fails) and then re-run *this* cell —
do not re-run the smoke test.

CDM stays disabled (`--include-cdm` omitted): it needs a Linux TeX Live +
ImageMagick + Ghostscript toolchain this environment has not been confirmed
to have. What prints is whatever the evaluator actually computed without it
— text edit distance, table TEDS, reading order — never a synthesised
overall score.

In [50]:
assert EVALUATOR_READY, "run the EVALUATOR ENVIRONMENT section first"

!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend baseline \
    --output {OUTPUT_ROOT}/baseline_smoke \
    --config {PIPELINE_CONFIG} \
    --omnidoc-python {OMNIDOC_PYTHON} \
    --skip-prepare

ground truth: /kaggle/working/doc-extraction/.external/OmniDocBench/demo_data/omnidocbench_demo/OmniDocBench_demo.json
predictions: /kaggle/working/results/baseline_smoke/predictions (4 files)
evaluator config: /kaggle/working/results/baseline_smoke/evaluator_config.yaml
invoking official evaluator (log: /kaggle/working/doc-extraction/experiments/005_omnidocbench/logs/baseline_smoke_evaluate.log) ...
evaluator exited with code 1 — see /kaggle/working/doc-extraction/experiments/005_omnidocbench/logs/baseline_smoke_evaluate.log
--- last stderr lines ---
Traceback (most recent call last):
  File "/kaggle/working/doc-extraction/.external/OmniDocBench/pdf_validation.py", line 1, in <module>
    from src.cli import main
  File "/kaggle/working/doc-extraction/.external/OmniDocBench/src/__init__.py", line 1, in <module>
    from .cli import main
  File "/kaggle/working/doc-extraction/.external/OmniDocBench/src/cli.py", line 1, in <module>
    from src.core.pipeline import build_save_name, load

In [51]:
print("==================================================")
print("10. 18-PAGE DEMO RUN (optional)")
print("==================================================")

10. 18-PAGE DEMO RUN (optional)


Prepare + evaluate the full 18-page official demo set — a slightly larger
correctness check than the 3-page smoke test, and a better throughput
sample. **Still not the full benchmark**: it always targets the bundled demo
set explicitly, whatever `DATASET_PATH` resolved to.

In [52]:
assert EVALUATOR_READY, "run the EVALUATOR ENVIRONMENT section first"
_demo = f"{OMNIDOC_REPO}/demo_data/omnidocbench_demo"

!python experiments/005_omnidocbench/run.py \
    --dataset {_demo} \
    --backend baseline \
    --output {OUTPUT_ROOT}/baseline_demo18 \
    --config {PIPELINE_CONFIG} \
    --omnidoc-python {OMNIDOC_PYTHON} \
    --match-workers 2

dataset: /kaggle/working/doc-extraction/.external/OmniDocBench/demo_data/omnidocbench_demo/OmniDocBench_demo.json
samples: 18 of 18 available (backend=baseline)
09:11:40 [INFO] [yanbaopptmerge_SE05.pdf_7-660a2591] render backend=pymupdf page=0 status=success runtime=0.29175611500068044
Loading weights: 100%|██████████████████████| 770/770 [00:00<00:00, 2834.51it/s]
09:11:59 [INFO] [yanbaopptmerge_SE05.pdf_7-660a2591] layout backend=docling page=0 status=success runtime=19.12549931799913
09:11:59 [INFO] [yanbaopptmerge_SE05.pdf_7-660a2591] ocr backend=docling page=0 status=success runtime=0.0008429310000792611
Loading weights: 100%|██████████████████████| 367/367 [00:00<00:00, 2691.14it/s]
[transformers] TableTransformerForObjectDetection LOAD REPORT from: microsoft/table-transformer-detection
Key                                                                         | Status     |  | 
----------------------------------------------------------------------------+------------+--+-
model.

In [53]:
print("==================================================")
print("11. FULL BENCHMARK (1651 pages)")
print("==================================================")

11. FULL BENCHMARK (1651 pages)


**Guarded by `USING_FULL_DATASET`** — refuses to run against the demo set.

Budget the time before starting. At the pre-fix CPU rate (~28 s/page) 1651
pages is ~13 h per backend, which does not fit a Kaggle session; that is
precisely why the GPU fixes matter. Check the measured s/page from section 8
and multiply by 1651 before committing to a run, and consider `--subset` for
a statistically useful slice instead of the whole set.

`--match-workers`: roughly 1/3-1/2 of the instance's CPU count (upstream's
own guidance, to avoid deadlocks/OOM in its worker pools). The evaluator is
CPU-bound regardless of the accelerator.

In [54]:
if not USING_FULL_DATASET:
    raise RuntimeError(
        "Refusing to run the full benchmark: no Kaggle Dataset is attached "
        "(USING_FULL_DATASET=False, DATASET_PATH is the 18-page demo set). "
        "Attach the full OmniDocBench dataset via 'Add Input', set "
        "KAGGLE_DATASET_NAME in the DATASET section, re-run that section, "
        "then re-run this cell."
    )
assert EVALUATOR_READY, "run the EVALUATOR ENVIRONMENT section first"

!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend baseline \
    --output {OUTPUT_ROOT}/baseline_full \
    --config {PIPELINE_CONFIG} \
    --omnidoc-python {OMNIDOC_PYTHON} \
    --match-workers 4

RuntimeError: Refusing to run the full benchmark: no Kaggle Dataset is attached (USING_FULL_DATASET=False, DATASET_PATH is the 18-page demo set). Attach the full OmniDocBench dataset via 'Add Input', set KAGGLE_DATASET_NAME in the DATASET section, re-run that section, then re-run this cell.

In [ ]:
if not USING_FULL_DATASET:
    raise RuntimeError("Refusing to run: no Kaggle Dataset attached — see the cell above.")
assert EVALUATOR_READY, "run the EVALUATOR ENVIRONMENT section first"

!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend docling \
    --output {OUTPUT_ROOT}/docling_full \
    --config {PIPELINE_CONFIG} \
    --omnidoc-python {OMNIDOC_PYTHON} \
    --match-workers 4

In [ ]:
print("==================================================")
print("12. RESULTS & METRICS")
print("==================================================")

In [ ]:
from pathlib import Path

for d in sorted(Path(OUTPUT_ROOT).glob("*")):
    report = d / "report.md"
    if report.exists():
        print(f"===== {d.name} =====")
        print(report.read_text(encoding="utf-8"))
        print()

In [ ]:
print("==================================================")
print("13. PACKAGE RESULTS")
print("==================================================")

`/kaggle/working` persists for the session and downloads from the output
panel. To fold results back into the repo, copy just the small
committed-shape files (not `predictions/`, not the evaluator's raw debug
dumps) into `experiments/005_omnidocbench/results/<backend>/`.

In [ ]:
from pathlib import Path

KEEP = ("report.md", "metrics.json", "runtime.json", "run_metadata.json")

for d in sorted(Path(OUTPUT_ROOT).glob("*")):
    for name in KEEP:
        f = d / name
        if f.exists():
            print(f"kept for output panel: {f}")

print()
print("Download these from the Kaggle output panel, then copy locally into")
print("experiments/005_omnidocbench/results/<backend>/.")

In [ ]:
print("==================================================")
print("14. REPRODUCIBILITY METADATA")
print("==================================================")

In [ ]:
import subprocess
from datetime import datetime, timezone
from pathlib import Path

def _rev(path):
    return subprocess.run(["git", "-C", path, "rev-parse", "HEAD"],
                          capture_output=True, text=True).stdout.strip()

print(f"Timestamp (UTC)       : {datetime.now(timezone.utc).isoformat()}")
print(f"doc-extraction commit : {_rev(REPO_DIR)}")
print(f"OmniDocBench commit   : {_rev(OMNIDOC_REPO)} (pinned {PINNED_OMNIDOC_COMMIT})")
print(f"Kernel Python         : {sys.version.split()[0]}")
print(f"Evaluator interpreter : {OMNIDOC_PYTHON}")
print(f"CUDA available        : {CUDA_AVAILABLE}")
print(f"Pipeline config       : {PIPELINE_CONFIG} (device={_cfg.device})")
print(f"USING_FULL_DATASET    : {USING_FULL_DATASET}")
print(f"DATASET_PATH          : {DATASET_PATH}")